# HW2 Part 1: Data Cleaning & Validation

Cleans `ohlc_hw` from the local working copy `data/data.db` and saves `ohlc_hw_cleaned` to the same file; the raw table is never modified. The supplied `ohlc` table is used as the trading calendar and as the source of date-specific adjustment factors.

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

DB_PATH = "data/data.db"
PRICE_COLS = ["open", "high", "low", "close"]
VALUE_COLS = PRICE_COLS + ["adj_close", "volume"]
REPAIR_FLAGS = ["price_filled", "price_interpolated", "adj_close_rebuilt", "volume_filled"]

## 1.1 Load and Inspect

In [2]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
    ref = pd.read_sql_query("SELECT * FROM ohlc", conn)

ref["date"] = pd.to_datetime(ref["ts"], format="%Y-%m-%d")
ref["ref_factor"] = ref["adj_close"] / ref["close"]

raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


In [3]:
raw.describe()

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


| Column | Meaning |
|---|---|
| `ts` | Observation date (text, `YYYY-MM-DD`) |
| `ticker` | Stock symbol (dirty: contains HTML markup) |
| `open`, `high`, `low`, `close` | Raw daily prices |
| `adj_close` | Vendor close adjusted for splits and dividends |
| `volume` | Shares traded |

In [4]:
dates = pd.to_datetime(raw["ts"], format="%Y-%m-%d", errors="coerce")
print(f"Unparseable dates: {dates.isna().sum()}")
print(f"Date range: {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}")
print(f"Distinct raw ticker labels: {raw['ticker'].nunique()}")
raw["ticker"].drop_duplicates().head(6).tolist()

Unparseable dates: 0
Date range: 1962-01-02 to 2024-12-06
Distinct raw ticker labels: 150


['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ']

### Ticker normalization and duplicates

Each stock appears under five labels: plain and four HTML-styled variants. Tickers are normalized before any grouping or deduplication, and the original label is kept in `raw_ticker`. Exact duplicates are dropped; `(ticker, date)` keys whose rows still disagree are quarantined instead of keeping one arbitrarily.

In [5]:
def normalize_tickers(data: pd.DataFrame) -> pd.DataFrame:
    """Strip HTML markup, whitespace and case from tickers; keep the original label."""
    out = data.rename(columns={"ticker": "raw_ticker"})
    out["ticker"] = out["raw_ticker"].str.replace(r"<[^>]*>", "", regex=True).str.strip().str.upper()
    out["date"] = pd.to_datetime(out["ts"], format="%Y-%m-%d")
    return out


def deduplicate(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Drop exact duplicates; split off (ticker, date) keys whose rows still disagree."""
    data = data.drop_duplicates(subset=["ticker", "date", *VALUE_COLS])
    conflict = data.duplicated(["ticker", "date"], keep=False)
    return data[~conflict], data[conflict]

In [6]:
df = normalize_tickers(raw)
print("Raw labels per ticker:", df.groupby("ticker")["raw_ticker"].nunique().value_counts().to_dict())
print(f"Tickers ({df['ticker'].nunique()}):", ", ".join(sorted(df["ticker"].unique())))

df, conflicts = deduplicate(df)
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
n_duplicates = len(raw) - len(df) - len(conflicts)
print(f"Exact duplicates removed: {n_duplicates:,}")
print(f"Conflicting (ticker, date) rows: {len(conflicts)}")
print(f"Rows after deduplication: {len(df):,}")

Raw labels per ticker: {5: 30}
Tickers (30): AAPL, AMGN, AMZN, AXP, BA, CAT, CRM, CSCO, CVX, DIS, GS, HD, HON, IBM, JNJ, JPM, KO, MCD, MMM, MRK, MSFT, NKE, NVDA, PG, SHW, TRV, UNH, V, VZ, WMT
Exact duplicates removed: 29,460
Conflicting (ticker, date) rows: 0
Rows after deduplication: 368,251


## 1.2 Detect Data Quality Issues

### Missing data

Missing trading days use each ticker's dates in `ohlc` as the reference calendar, over that ticker's first–last date in `ohlc_hw` (business weekdays would count exchange holidays as gaps). Missing dates are reported, not filled with invented prices.

In [7]:
def find_missing_trading_days(data: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    """Return (ticker, date) pairs in `calendar` but absent from `data`,
    checked only within each ticker's first-to-last date in `data`."""
    observed = data[["ticker", "date"]].drop_duplicates()
    span = observed.groupby("ticker")["date"].agg(first="min", last="max")
    expected = calendar.merge(span, on="ticker")
    expected = expected[expected["date"].between(expected["first"], expected["last"])]
    merged = expected[["ticker", "date"]].merge(observed, how="left", indicator=True)
    return merged.loc[merged["_merge"] == "left_only", ["ticker", "date"]].reset_index(drop=True)


def summarize_missing_days(data: pd.DataFrame, missing: pd.DataFrame) -> list[str]:
    """'Ticker X is missing Y trading days between [start] and [end]' for each ticker."""
    span = data.groupby("ticker")["date"].agg(["min", "max"])
    counts = missing["ticker"].value_counts()
    return [f"Ticker {t} is missing {counts.get(t, 0)} trading days between "
            f"{r['min']:%Y-%m-%d} and {r['max']:%Y-%m-%d}" for t, r in span.iterrows()]


calendar = ref[["ticker", "date"]]
print("Missing values per column:")
print(df[VALUE_COLS].isna().sum().to_string())

missing_days = find_missing_trading_days(df, calendar)
print(f"\nTotal missing trading days: {len(missing_days)}")
print("\n".join(summarize_missing_days(df, missing_days)))

Missing values per column:
open             0
high             0
low              0
close            0
adj_close    18374
volume           0



Total missing trading days: 0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 

In [8]:
# The database has no gaps, so show the detector on a tiny calendar with one date removed.
demo_calendar = pd.DataFrame({"ticker": "DEMO", "date": pd.to_datetime(
    ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08"])})
demo_data = demo_calendar.drop(index=2)

demo_missing = find_missing_trading_days(demo_data, demo_calendar)
print(summarize_missing_days(demo_data, demo_missing))
assert demo_missing["date"].tolist() == [pd.Timestamp("2024-01-04")]

['Ticker DEMO is missing 1 trading days between 2024-01-02 and 2024-01-08']


### Outliers

Simple returns on raw close within each ticker. A return is flagged when `|return - mean| > 3 * std` (sample std, `ddof=1`); undefined returns and tickers with zero or undefined std are never flagged. A flag counts as a confirmed **data error** only when the price was printed on zero volume and the next day's flagged return reverses it (two-day move within ±5%); all other flags are review flags.

In [9]:
def flag_return_outliers(data: pd.DataFrame) -> pd.Series:
    """True where a return is more than 3 sample std from its ticker's mean return."""
    grouped = data.groupby("ticker")["return"]
    mean, std = grouped.transform("mean"), grouped.transform("std")
    return data["return"].notna() & std.gt(0) & (data["return"] - mean).abs().gt(3 * std)


def flag_spike_errors(data: pd.DataFrame) -> pd.Series:
    """Zero-volume outliers fully reversed by an outlier on the next day: one-day bad prints."""
    next_return = data.groupby("ticker")["return"].shift(-1)
    next_flagged = data.groupby("ticker")["return_outlier"].shift(-1, fill_value=False)
    round_trip = ((1 + data["return"]) * (1 + next_return) - 1).abs() < 0.05
    return data["return_outlier"] & data["volume"].eq(0) & next_flagged & round_trip


df["return"] = df.groupby("ticker")["close"].pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
df["return_outlier"] = flag_return_outliers(df)
df["spike_error"] = flag_spike_errors(df)
print(f"3-sigma return flags: {df['return_outlier'].sum():,}")
print(f"Confirmed one-day spikes: {df['spike_error'].sum()}")

compare = df.merge(ref, on=["ticker", "date"], suffixes=("", "_ref"))
raw_differs = ~np.isclose(compare[PRICE_COLS], compare[[f"{c}_ref" for c in PRICE_COLS]]).all(axis=1)
print(f"Rows whose raw OHLC differs from the ohlc reference: {raw_differs.sum()}")

spike_rows = df.index[df["spike_error"]]
df.loc[spike_rows.union(spike_rows - 1).union(spike_rows + 1),
       ["ticker", "ts", "open", "high", "low", "close", "volume", "return"]]

3-sigma return flags: 5,056
Confirmed one-day spikes: 4
Rows whose raw OHLC differs from the ohlc reference: 0


,ticker,ts,open,high,low,close,volume,return
28585,AXP,1972-12-13,4.59,4.59,4.59,4.59,844816,-0.008639
28586,AXP,1972-12-14,4.01,4.01,4.01,4.01,0,-0.126362
28587,AXP,1972-12-15,4.60,4.60,4.60,4.60,979282,0.147132
318331,TRV,1976-10-27,4.61,4.61,4.61,4.61,230400,0.024444
318332,TRV,1976-10-28,9.22,9.22,9.22,9.22,0,1.000000
318333,TRV,1976-10-29,4.69,4.69,4.69,4.69,79200,-0.491323
318872,TRV,1978-12-19,4.02,4.02,4.02,4.02,72000,0.000000
318873,TRV,1978-12-20,8.03,8.03,8.03,8.03,0,0.997512
318874,TRV,1978-12-21,4.09,4.09,4.09,4.09,105600,-0.490660
319537,TRV,1981-08-07,6.03,6.03,6.03,6.03,264800,-0.004950


Raw OHLC matches the vendor `ohlc` table on every row, so the 3σ moves are not transcription errors in this file. They are kept as review flags; their causes (news, corporate actions) are not verified here, and raw prices are already split-adjusted (no drop at AAPL's 2020 split). The four confirmed spikes (three TRV days at double the price, one AXP day 13% low) are flat bars printed on zero volume that fully reverse the next day, so they are treated as data errors.

### Inconsistencies

In [10]:
def price_issue_masks(data: pd.DataFrame) -> dict[str, pd.Series]:
    """Boolean masks for impossible or suspicious price and volume values."""
    prices = data[PRICE_COLS + ["adj_close"]]
    return {
        "high_below_close": data["high"] < data["close"],
        "low_above_close": data["low"] > data["close"],
        "open_outside_range": (data["open"] < data["low"]) | (data["open"] > data["high"]),
        "nonpositive_or_infinite_price": (prices.le(0) | np.isinf(prices)).any(axis=1),
        "adj_close_inconsistent": data["adj_close"].notna() & data["ref_factor"].notna()
            & ~np.isclose(data["adj_close"], data["close"] * data["ref_factor"], rtol=1e-6),
        "zero_volume": data["volume"].eq(0),
    }


df = df.merge(ref[["ticker", "date", "ref_factor"]], on=["ticker", "date"], how="left")
masks = price_issue_masks(df)
pd.Series({name: mask.sum() for name, mask in masks.items()}, name="rows")

high_below_close                     0
low_above_close                      0
open_outside_range                   5
nonpositive_or_infinite_price        5
adj_close_inconsistent           10417
zero_volume                         75
Name: rows, dtype: int64

`adj_close_inconsistent` compares vendor `adj_close` with `close × (adj_close / close)` from the matching `ohlc` row. The 10k+ inconsistent values are 4–24× too large; the reference factor never exceeds 1.

### Data Quality Report

| Issue | Severity | Why |
|---|---|---|
| Missing raw OHLC, nonpositive/infinite price, `high < close`, `low > close`, open outside range | high | Impossible or unusable prices break returns and validation |
| `adj_close` inconsistent with reference factor | high | Corrupts every report return |
| Confirmed one-day spike | high | Creates a false return that reverses the next day |
| Conflicting `(ticker, date)` rows | high | True value unknown |
| Missing `adj_close` | medium | Needed for reports, but recoverable from raw close |
| Missing trading day | medium | No price lost, but the next return spans two days |
| Missing volume, zero volume | low | Not used in risk metrics; zero volume may mean a stale quote |
| 3σ return (review flag) | low | Matches the vendor reference; kept |

In [11]:
SEVERITY = {
    **{f"missing_{c}": "high" for c in PRICE_COLS},
    "missing_adj_close": "medium", "missing_volume": "low", "missing_trading_day": "medium",
    "conflicting_duplicate": "high", "high_below_close": "high", "low_above_close": "high",
    "open_outside_range": "high", "nonpositive_or_infinite_price": "high",
    "adj_close_inconsistent": "high", "zero_volume": "low", "spike_error": "high", "return_outlier": "low",
}


def make_issues(data: pd.DataFrame, issue_type: str, description, mask=None) -> pd.DataFrame:
    """Quality Report rows for the rows of `data` selected by `mask` (all rows if None)."""
    rows = data if mask is None else data[mask]
    if isinstance(description, pd.Series):
        description = description.loc[rows.index]
    return pd.DataFrame({"ticker": rows["ticker"], "issue_type": issue_type, "issue_date": rows["date"],
                         "description": description, "severity": SEVERITY[issue_type]})


return_text = df["return"].map("{:+.1%}".format)
quality_report = pd.concat([
    *(make_issues(df, f"missing_{c}", f"{c} is missing", df[c].isna()) for c in VALUE_COLS),
    make_issues(missing_days, "missing_trading_day", "No row for a date in the ohlc calendar"),
    make_issues(conflicts, "conflicting_duplicate", "Rows disagree for the same (ticker, date)"),
    *(make_issues(df, name, name.replace("_", " "), mask) for name, mask in masks.items()),
    make_issues(df, "spike_error", "Return " + return_text + " reversed next day", df["spike_error"]),
    make_issues(df, "return_outlier", "3-sigma return " + return_text + "; matches reference, kept",
                df["return_outlier"] & ~df["spike_error"]),
], ignore_index=True).sort_values(["ticker", "issue_date"], ignore_index=True)

print(f"Issues: {len(quality_report):,}")
display(quality_report.groupby(["severity", "issue_type"]).size().rename("count").to_frame())
quality_report.head()

Issues: 33,932


count
severity issue_type                          
high     adj_close_inconsistent         10417
         nonpositive_or_infinite_price      5
         open_outside_range                 5
         spike_error                        4
low      return_outlier                  5052
         zero_volume                       75
medium   missing_adj_close              18374

,ticker,issue_type,issue_date,description,severity
0,AAPL,return_outlier,1980-12-26,"3-sigma return +9.2%; matches reference, kept",low
1,AAPL,missing_adj_close,1981-01-08,adj_close is missing,medium
2,AAPL,missing_adj_close,1981-01-13,adj_close is missing,medium
3,AAPL,adj_close_inconsistent,1981-02-03,adj close inconsistent,high
4,AAPL,missing_adj_close,1981-02-12,adj_close is missing,medium


## 1.3 Clean the Data

Repairs, in order:
1. Drop rows with all raw prices missing (before any fill).
2. Invalid raw cells (nonpositive, infinite, open or close outside `[low, high]`) → NaN, then forward-fill within the ticker (Option 1). Rows with no earlier value to fill from are quarantined.
3. `adj_close` missing or inconsistent → rebuilt as validated `close ×` that date's `ohlc` factor. Factors are never carried across dates.
4. Confirmed spikes: `close` and `adj_close` interpolated between the neighbouring days; open/high/low set to the new close (the original bars were flat). The reference factor is not used on these dates (it also spikes on the TRV days).
5. Missing volume → 0 with a `volume_filled` flag (unknown, not "no trading").
6. Rows still failing validation are quarantined.

Zero-volume days and review-flagged returns are kept.

In [12]:
def invalid_rows(data: pd.DataFrame) -> pd.Series:
    """True where a row breaks any rule for retained data."""
    prices = data[PRICE_COLS + ["adj_close"]]
    return (~np.isfinite(prices).all(axis=1)
            | prices.le(0).any(axis=1)
            | data["low"].gt(data[["open", "close"]].min(axis=1))
            | data["high"].lt(data[["open", "close"]].max(axis=1))
            | ~np.isfinite(data["volume"])
            | data["volume"].lt(0))


def clean(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply the repairs to ticker/date-sorted data; return (cleaned, quarantined)."""
    data = data.copy()
    set_aside = []

    def quarantine(mask, reason):
        set_aside.append(data[mask].assign(reason=reason))
        return data[~mask]

    data = quarantine(data[PRICE_COLS].isna().all(axis=1), "all raw prices missing")

    prices = data[PRICE_COLS]
    bad = prices.le(0) | np.isinf(prices)
    bad["open"] |= (data["open"] < data["low"]) | (data["open"] > data["high"])
    bad["close"] |= (data["close"] < data["low"]) | (data["close"] > data["high"])
    data["price_filled"] = (bad | prices.isna()).any(axis=1)
    data[PRICE_COLS] = prices.mask(bad).groupby(data["ticker"]).ffill()
    data = quarantine(data[PRICE_COLS].isna().any(axis=1), "invalid raw price with no earlier value")

    spike = data["spike_error"]
    data.loc[spike, "ref_factor"] = np.nan
    consistent = np.isclose(data["adj_close"], data["close"] * data["ref_factor"], rtol=1e-6)
    data["adj_close_rebuilt"] = ~spike & ~consistent
    data.loc[data["adj_close_rebuilt"], "adj_close"] = data["close"] * data["ref_factor"]

    data["price_interpolated"] = spike
    data.loc[spike, ["close", "adj_close"]] = np.nan
    interpolated = data.groupby("ticker")[["close", "adj_close"]].transform(
        lambda s: s.interpolate(limit_area="inside"))
    data.loc[spike, ["close", "adj_close"]] = interpolated[spike]
    data.loc[spike, ["open", "high", "low"]] = data.loc[spike, ["close"] * 3].to_numpy()

    data["volume_filled"] = data["volume"].isna()
    data["volume"] = data["volume"].fillna(0)

    data = quarantine(invalid_rows(data), "still invalid after repair")
    return data, pd.concat(set_aside, ignore_index=True)

In [13]:
cleaned, quarantined = clean(df)
quarantined = pd.concat([conflicts.assign(reason="conflicting values for (ticker, date)"), quarantined],
                        ignore_index=True)
assert len(raw) == n_duplicates + len(quarantined) + len(cleaned)

print(f"Raw rows:                    {len(raw):,}")
print(f"Exact duplicates removed:    {n_duplicates:,}")
print(f"Quarantined:                 {len(quarantined):,}")
print(f"Retained in ohlc_hw_cleaned: {len(cleaned):,}")
print("\nRepair flags:")
print(cleaned[REPAIR_FLAGS].sum().to_string())
quarantined[["ticker", "ts", *VALUE_COLS, "reason"]]

Raw rows:                    397,711
Exact duplicates removed:    29,460
Quarantined:                 5
Retained in ohlc_hw_cleaned: 368,246

Repair flags:
price_filled              0
price_interpolated        4
adj_close_rebuilt     28790
volume_filled             0


,ticker,ts,open,high,low,close,adj_close,volume,reason
0,CVX,1962-01-02,NaN,3.30000,3.24000,3.30000,0.335750,105840,invalid raw price with no earlier value
1,JNJ,1962-01-02,NaN,0.22338,0.22222,0.22338,0.064012,0,invalid raw price with no earlier value
2,MCD,1966-07-05,NaN,0.27366,0.26749,0.26955,0.118090,388800,invalid raw price with no earlier value
3,MRK,1962-01-02,NaN,0.39703,0.39317,0.39537,0.068447,633830,invalid raw price with no earlier value
4,PG,1962-01-02,NaN,1.44000,1.42000,1.43000,0.259300,192000,invalid raw price with no earlier value


In [14]:
# Repaired spike rows next to their neighbours
around = cleaned.index[cleaned["price_interpolated"]]
cleaned.loc[around.union(around - 1).union(around + 1), ["ticker", "ts", "close", "adj_close", "price_interpolated"]]

,ticker,ts,close,adj_close,price_interpolated
28585,AXP,1972-12-13,4.590,1.550000,False
28586,AXP,1972-12-14,4.595,1.555000,True
28587,AXP,1972-12-15,4.600,1.560000,False
318331,TRV,1976-10-27,4.610,0.980460,False
318332,TRV,1976-10-28,4.650,0.988765,True
318333,TRV,1976-10-29,4.690,0.997070,False
318872,TRV,1978-12-19,4.020,0.906450,False
318873,TRV,1978-12-20,4.055,0.915270,True
318874,TRV,1978-12-21,4.090,0.924090,False
319537,TRV,1981-08-07,6.030,1.570000,False


**Injected example.** The database has no missing raw prices, missing volume, `close > high`, infinite values or conflicting keys, so the detectors and repairs are shown on a small frame containing each.

In [15]:
demo = pd.DataFrame({
    "ticker": "DEMO",
    "date": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05",
                            "2024-01-08", "2024-01-09", "2024-01-09"]),
    "open":      [0.0, 10.0, np.nan, 10.0, np.nan, 10.0, 10.0],     # leading zero; missing open
    "high":      [11.0, 11.0, np.nan, 11.0, 11.0, 11.0, 11.0],
    "low":       [9.0, 9.0, np.nan, 9.0, 9.0, 9.0, 9.0],
    "close":     [10.0, 10.5, np.nan, 12.0, np.inf, 10.0, 10.2],    # close > high; infinite close
    "adj_close": [9.0, np.nan, np.nan, 10.8, 9.0, 9.0, 9.18],       # missing; inconsistent
    "volume":    [100.0, np.nan, 100.0, 100.0, 100.0, 100.0, 100.0],  # missing volume
    "ref_factor": 0.9,
    "spike_error": False,
})
display(pd.DataFrame(price_issue_masks(demo)).assign(date=demo["date"]))

demo_dedup, demo_conflicts = deduplicate(demo)
demo_clean, demo_quarantined = clean(demo_dedup)
display(demo_clean[["date", *VALUE_COLS, *REPAIR_FLAGS]], demo_quarantined[["date", "reason"]])

assert len(demo_conflicts) == 2
assert demo_clean["date"].dt.day.tolist() == [3, 5, 8]
assert demo_clean["close"].tolist() == [10.5, 10.5, 10.5]
assert np.allclose(demo_clean["adj_close"], 10.5 * 0.9)
assert demo_clean["volume_filled"].tolist() == [True, False, False]
assert not invalid_rows(demo_clean).any()

,high_below_close,low_above_close,open_outside_range,nonpositive_or_infinite_price,adj_close_inconsistent,zero_volume,date
0,False,False,True,True,False,False,2024-01-02
1,False,False,False,False,False,False,2024-01-03
2,False,False,False,False,False,False,2024-01-04
3,True,False,False,False,False,False,2024-01-05
4,True,False,False,True,True,False,2024-01-08
5,False,False,False,False,False,False,2024-01-09
6,False,False,False,False,False,False,2024-01-09


,date,open,high,low,close,adj_close,volume,price_filled,price_interpolated,adj_close_rebuilt,volume_filled
1,2024-01-03,10.0,11.0,9.0,10.5,9.45,0.0,False,False,True,True
3,2024-01-05,10.0,11.0,9.0,10.5,9.45,100.0,True,False,True,False
4,2024-01-08,10.0,11.0,9.0,10.5,9.45,100.0,True,False,True,False


,date,reason
0,2024-01-04,all raw prices missing
1,2024-01-02,invalid raw price with no earlier value


### Validation

In [16]:
checks = {
    "No missing/non-finite/nonpositive prices, valid OHLC bounds, finite nonnegative volume":
        not invalid_rows(cleaned).any(),
    "Unique (ticker, date) keys": not cleaned.duplicated(["ticker", "date"]).any(),
    "adj_close = close x ohlc factor wherever the factor is trusted":
        not price_issue_masks(cleaned)["adj_close_inconsistent"].any(),
    "No missing trading days within retained history": find_missing_trading_days(cleaned, calendar).empty,
}
for name, passed in checks.items():
    print("PASS" if passed else "FAIL", name)
assert all(checks.values())

PASS No missing/non-finite/nonpositive prices, valid OHLC bounds, finite nonnegative volume
PASS Unique (ticker, date) keys
PASS adj_close = close x ohlc factor wherever the factor is trusted
PASS No missing trading days within retained history


In [17]:
def issue_counts(data: pd.DataFrame) -> pd.Series:
    """Missing-value and price-issue counts used for the before/after comparison."""
    missing = data[VALUE_COLS].isna().sum().add_prefix("missing_")
    return pd.concat([missing, pd.Series({k: m.sum() for k, m in price_issue_masks(data).items()})])


display(pd.DataFrame({"before": issue_counts(df), "after": issue_counts(cleaned)}))
stats = ["count", "mean", "min", "max"]
pd.concat({"before": df[VALUE_COLS].describe().T[stats], "after": cleaned[VALUE_COLS].describe().T[stats]},
          axis=1)

,before,after
missing_open,0,0
missing_high,0,0
missing_low,0,0
missing_close,0,0
missing_adj_close,18374,0
missing_volume,0,0
high_below_close,0,0
low_above_close,0,0
open_outside_range,5,0
nonpositive_or_infinite_price,5,0


before                                           after  \
              count          mean       min           max     count   
open       368251.0  4.630877e+01  0.000000  6.042600e+02  368246.0   
high       368251.0  4.677120e+01  0.005208  6.086300e+02  368246.0   
low        368251.0  4.583977e+01  0.004801  5.978800e+02  368246.0   
close      368251.0  4.631671e+01  0.005208  6.054000e+02  368246.0   
adj_close  349877.0  5.171154e+01  0.002945  1.044251e+04  368246.0   
volume     368251.0  3.095142e+07  0.000000  9.276606e+09  368246.0   

                                                 
                   mean       min           max  
open       4.630936e+01  0.005208  6.042600e+02  
high       4.677179e+01  0.005208  6.086300e+02  
low        4.584034e+01  0.004801  5.978800e+02  
close      4.631729e+01  0.005208  6.054000e+02  
adj_close  3.727465e+01  0.002945  6.054000e+02  
volume     3.095184e+07  0.000000  9.276606e+09

Zero-volume days remain by design (kept and flagged in the Quality Report).

### Save

In [18]:
save_cols = ["ts", "ticker", "raw_ticker", *VALUE_COLS, "return_outlier", *REPAIR_FLAGS]
output = cleaned.assign(ts=cleaned["date"].dt.strftime("%Y-%m-%d"))[save_cols]

with closing(sqlite3.connect(DB_PATH)) as conn:
    output.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
    saved = pd.read_sql_query("SELECT COUNT(*) AS rows, COUNT(DISTINCT ticker) AS tickers, "
                              "MIN(ts) AS first, MAX(ts) AS last FROM ohlc_hw_cleaned", conn)
saved

,rows,tickers,first,last
0,368246,30,1962-01-02,2024-12-06
